# Model: patronus-studio/wolf-defender-prompt-injection

**main-v3**: main-v2 with two changes, trained together as variant `wolf1024_ls`:
- **max_len 1024** instead of 384 (wolf-defender reads up to 8k tokens). Batch 8 × 8 accumulation keeps the effective batch at 64. The training log prints how many prompts are over 384 tokens and how many are still cut at 1024.
- **Label smoothing 0.05** in the loss.

Everything else is main-v2: 5-fold CV, class offsets tuned on the OOF, both T4s, the collapse guard and blending.

**How to run:** GPU T4×2, Internet off. Attach the competition data and the `wolf-defender-prompt-injection` model, then Run All. About 60–90 min. If a fold runs out of GPU memory, set `GRAD_CKPT = True` in the next cell and rerun.

**Blend with main-v2:** also attach the output of the main-v2 run before Run All. The merge cell then picks up both `wolf384` and `wolf1024_ls`, the blend cell picks their weights on the OOF, and `submission.csv` is the blend. Blend only runs of this one backbone (one approved model per solution).

In [ ]:
%%writefile train.py
# One process per GPU: `python train.py 0 2 4` trains those folds and writes their per-fold files.
import gc, glob, json, math, os, re, sys, time
import numpy as np, pandas as pd, torch, transformers
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

# ---- config: one variant per run; main-v2's recipe with max_len 1024 and label smoothing ---------
# Extra variants to blend (same backbone only): change VARIANT and one setting, e.g. SEED = 43 or LLRD = 0.9
VARIANT = "wolf1024_ls"
MAX_LEN, TRUNCATION, HEAD_FRAC = 1024, "headtail", 2 / 3  # TRUNCATION: "head" or "headtail"
LR, LLRD, EPOCHS, SEED = 3e-5, 1.0, 4, 42
BS, ACCUM, WARMUP, WD = 8, 8, 0.1, 0.01             # effective batch = BS * ACCUM = 64
LANG_PREFIX, GRAD_CKPT, LABEL_SMOOTH = False, False, 0.05  # GRAD_CKPT = True if a fold runs out of GPU memory
MODEL_NAME, LABELS, OUT = "wolf-defender", ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"], "/kaggle/working"
CONFIG = {k: v for k, v in dict(globals()).items() if k.isupper()}

transformers.logging.set_verbosity_error()  # hides the harmless "LOAD REPORT" and long-prompt warnings
dev = torch.device("cuda")
DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
models = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True) if MODEL_NAME in p.lower()]
assert models, f"attach the {MODEL_NAME} model as an input"
assert MAX_LEN <= AutoConfig.from_pretrained(models[0], local_files_only=True).max_position_embeddings, "MAX_LEN is longer than this model reads"

train, test = pd.read_csv(f"{DATA_DIR}/train.csv"), pd.read_csv(f"{DATA_DIR}/test.csv")
y = train.label.map({l: i for i, l in enumerate(LABELS)}).values
fold = np.zeros(len(train), dtype=int)
for k, (_, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(train, train.label + "_" + train.language)):
    fold[va] = k
tok = AutoTokenizer.from_pretrained(models[0], local_files_only=True)
core, full = tok("hello world", add_special_tokens=False)["input_ids"], tok("hello world")["input_ids"]
cut = next(i for i in range(len(full)) if full[i:i + len(core)] == core)
PRE, POST = full[:cut], full[cut + len(core):]  # this tokenizer's own special tokens, e.g. [CLS] ... [SEP]

def encode(df):
    texts = "[" + df.language + "] " + df.text if LANG_PREFIX else df.text
    budget = MAX_LEN - len(PRE) - len(POST)
    head = int(budget * HEAD_FRAC) if TRUNCATION == "headtail" else budget
    # long prompts keep their first `head` and last `budget - head` tokens (no tail for plain head truncation)
    return [PRE + (x if len(x) <= budget else x[:head] + x[len(x) - budget + head:]) + POST
            for x in tok(list(texts), add_special_tokens=False)["input_ids"]]

X, X_te = encode(train), encode(test)
assert max(map(len, X + X_te)) <= MAX_LEN
print(f"prompts over 384 tokens: {np.mean([len(x) > 384 for x in X]):.1%} | still cut at {MAX_LEN}: {np.mean([len(x) == MAX_LEN for x in X]):.1%}", flush=True)

def pad(seqs):
    ids = torch.full((len(seqs), max(map(len, seqs))), tok.pad_token_id)
    mask = torch.zeros_like(ids)
    for r, s in enumerate(seqs):
        ids[r, :len(s)], mask[r, :len(s)] = torch.tensor(s), 1
    return {"input_ids": ids.to(dev), "attention_mask": mask.to(dev)}

def train_batches(idx, rng):
    # sort by length inside chunks of 50 batches: little padding, order still random
    perm, out = rng.permutation(idx), []
    for i in range(0, len(perm), BS * 50):
        c = perm[i:i + BS * 50]
        c = c[np.argsort([len(X[j]) for j in c])]
        out += [c[k:k + BS] for k in range(0, len(c), BS)]
    return [out[i] for i in rng.permutation(len(out))]

@torch.no_grad()
def predict(model, seqs, bs=32):
    model.eval()
    order, out = np.argsort([len(s) for s in seqs]), np.zeros((len(seqs), 3), dtype=np.float32)
    for i in range(0, len(seqs), bs):
        b = order[i:i + bs]
        with torch.autocast("cuda", dtype=torch.float16):
            out[b] = model(**pad([seqs[j] for j in b])).logits.float().softmax(-1).cpu().numpy()
    return out

def optimizer(model):
    n_layers = model.config.num_hidden_layers
    def lr_scale(name):  # layer-wise LR decay: head LR, top layer LR*LLRD, ..., embeddings LR*LLRD^(n_layers+1)
        m = re.search(r"(?:encoder\.layer|layers)\.(\d+)\.", name)  # XLM-R and ModernBERT layer names
        return LLRD ** (n_layers + 1) if "embeddings" in name else LLRD ** (n_layers - int(m.group(1))) if m else 1.0
    groups = {}
    for n, p in model.named_parameters():
        groups.setdefault((lr_scale(n), 0.0 if "bias" in n or "norm" in n.lower() else WD), []).append(p)
    return torch.optim.AdamW([{"params": ps, "lr": LR * s, "weight_decay": wd} for (s, wd), ps in groups.items()])

def run_fold(k, seed):
    gc.collect(); torch.cuda.empty_cache()
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    tr_idx, va_idx = np.where(fold != k)[0], np.where(fold == k)[0]
    model = AutoModelForSequenceClassification.from_pretrained(  # its 2-class head is replaced by a new 3-class one
        models[0], num_labels=3, ignore_mismatched_sizes=True, local_files_only=True).to(dev)
    if GRAD_CKPT:
        model.gradient_checkpointing_enable()
    w = 1 / np.sqrt(np.bincount(y[tr_idx], minlength=3) / len(tr_idx))
    loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(w / w.mean(), dtype=torch.float32, device=dev), label_smoothing=LABEL_SMOOTH)
    opt = optimizer(model)
    steps = EPOCHS * math.ceil(math.ceil(len(tr_idx) / BS) / ACCUM)  # optimizer steps
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler("cuda")
    langs, log, best_f1, t_start = train.language.values[va_idx], [], -1, time.time()
    print(f"[fold {k}] seed {seed}: {len(tr_idx)} train / {len(va_idx)} val rows", flush=True)
    for ep in range(1, EPOCHS + 1):
        model.train()
        t0, batches = time.time(), train_batches(tr_idx, rng)
        for i, b in enumerate(batches, 1):
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**pad([X[j] for j in b])).logits
            loss = loss_fn(logits.float(), torch.tensor(y[b], device=dev))
            scaler.scale(loss / ACCUM).backward()
            if i % ACCUM == 0 or i == len(batches):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            if i == 50 or i % 400 == 0:
                print(f"[fold {k}] ep{ep} step {i}/{len(batches)} loss {loss.item():.4f} | "
                      f"{(time.time() - t0) / i * (len(batches) - i) / 60:.1f} min left in epoch", flush=True)
        p = predict(model, [X[j] for j in va_idx])
        pred, yv = p.argmax(1), y[va_idx]
        f1 = f1_score(yv, pred, average="macro")
        log.append({"epoch": ep, "macro_f1": f1, "minutes": (time.time() - t_start) / 60,
                    "per_class_f1": dict(zip(LABELS, f1_score(yv, pred, average=None, labels=[0, 1, 2]).tolist())),
                    "per_lang_f1": {l: f1_score(yv[langs == l], pred[langs == l], average="macro") for l in np.unique(langs)}})
        print(f"[fold {k}] == epoch {ep}: val macro-F1 {f1:.4f} ({(time.time() - t0) / 60:.1f} min)", flush=True)
        if ep == 1 and f1 < 0.40:
            return None  # collapsed to (nearly) one class
        if f1 > best_f1:  # keep the best epoch's weights in CPU RAM, never on disk
            best_f1, best_ep, best_p = f1, ep, p
            best_state = {n: t.to("cpu", copy=True) for n, t in model.state_dict().items()}
    model.load_state_dict(best_state)
    np.savez(f"{OUT}/oof_{VARIANT}_f{k}.npz", ids=np.asarray(train.id, dtype=str)[va_idx], probs=best_p)
    np.savez(f"{OUT}/test_{VARIANT}_f{k}.npz", ids=np.asarray(test.id, dtype=str), probs=predict(model, X_te))
    with open(f"{OUT}/log_{VARIANT}_f{k}.json", "w") as f:
        json.dump({"variant": VARIANT, "fold": k, "seed": seed, "config": CONFIG, "epochs": log, "best_epoch": best_ep}, f, indent=1)
    print(f"[fold {k}] done: best epoch {best_ep}/{EPOCHS}, val macro-F1 {best_f1:.4f}", flush=True)
    return best_f1

folds_to_run = [int(a) for a in sys.argv[1:]]
if 0 in folds_to_run:  # only one process writes the shared fold file
    pd.DataFrame({"id": train.id, "fold": fold}).to_csv(f"{OUT}/folds.csv", index=False)
for k in folds_to_run:
    if run_fold(k, SEED) is None:
        print(f"[fold {k}] collapsed (epoch-1 macro-F1 < 0.40): retrying with seed {SEED + 1000}", flush=True)
        if run_fold(k, SEED + 1000) is None:
            sys.exit(f"[fold {k}] collapsed twice: stop and check the setup")

In [ ]:
import os, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less fragmentation with mixed batch lengths
if torch.cuda.device_count() >= 2:  # one process per T4: folds 0, 2, 4 and 1, 3
    !CUDA_VISIBLE_DEVICES=0 python -u train.py 0 2 4 & CUDA_VISIBLE_DEVICES=1 python -u train.py 1 3 & wait
else:
    !python -u train.py 0 1 2 3 4

In [ ]:
import glob, os
import numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, f1_score

LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
train, test = pd.read_csv(f"{DATA_DIR}/train.csv"), pd.read_csv(f"{DATA_DIR}/test.csv")
y = train.label.map({l: i for i, l in enumerate(LABELS)}).values

# per-fold files from this run, plus any attached as inputs (another account's folds, earlier runs)
paths = {os.path.basename(p): p for p in glob.glob("/kaggle/input/**/*_f[0-9].npz", recursive=True) + glob.glob("/kaggle/working/*_f[0-9].npz")}
for v in sorted({n[4:-7] for n in paths if n.startswith("oof_")}):
    oof = [np.load(paths[n]) for n in sorted(paths) if n.startswith(f"oof_{v}_f")]
    tst = [np.load(paths[n]) for n in sorted(paths) if n.startswith(f"test_{v}_f")]
    if len(oof) < 5 or len(tst) < 5:
        print(f"SKIP {v}: only {len(oof)} OOF and {len(tst)} test fold files, need 5 of each")
        continue
    ids = np.concatenate([d["ids"] for d in oof])
    assert len(ids) == len(set(ids)) == len(train) and set(ids) == set(train.id), f"{v}: OOF ids missing or duplicated"
    assert all((d["ids"] == np.asarray(test.id, dtype=str)).all() for d in tst), f"{v}: test rows out of order"
    P = pd.DataFrame(np.concatenate([d["probs"] for d in oof]), index=ids).loc[train.id].values
    np.save(f"/kaggle/working/oof_{v}.npy", P)
    np.save(f"/kaggle/working/test_{v}.npy", np.mean([d["probs"] for d in tst], axis=0))
    pred = P.argmax(1)
    print(f"== {v}: OOF macro-F1 {f1_score(y, pred, average='macro'):.4f} | per class",
          dict(zip(LABELS, f1_score(y, pred, average=None).round(4).tolist())))
    print("   per language:", {l: round(f1_score(y[m], pred[m], average="macro"), 4) for l, m in train.groupby("language").indices.items()})
    print(confusion_matrix(y, pred))

In [ ]:
import itertools
from sklearn.model_selection import StratifiedKFold

# every merged variant: this run's and any attached from earlier runs
oofs = {os.path.basename(p)[4:-4]: p for p in glob.glob("/kaggle/input/**/oof_*.npy", recursive=True) + glob.glob("/kaggle/working/oof_*.npy")}
names = sorted(oofs)
assert names, "no oof_<variant>.npy found: check the training output above"
P = np.stack([np.load(oofs[n]) for n in names])
T = np.stack([np.load(os.path.join(os.path.dirname(oofs[n]), f"test_{n}.npy")) for n in names])
macro = lambda yy, scores: f1_score(yy, scores.argmax(1), average="macro")

# blend weights on a 0.05 simplex grid; max() keeps the first best, and the most even blends come first
grid = np.round(np.arange(0, 1.001, 0.05), 2)
combos = sorted((w for w in itertools.product(grid, repeat=len(names)) if abs(sum(w) - 1) < 1e-6),
                key=lambda w: np.abs(np.array(w) - 1 / len(w)).sum())
W = max(combos, key=lambda w: macro(y, np.tensordot(w, P, 1)))
logp, logt = np.log(np.tensordot(W, P, 1) + 1e-9), np.log(np.tensordot(W, T, 1) + 1e-9)

def tune(lp, yy, step):
    # class offsets on log-probs; checked from (0, 0) outwards, so a tie keeps the smallest offsets
    g = np.round(np.arange(-1.5, 1.5001, step), 2)
    return max(sorted(itertools.product(g, g), key=lambda b: abs(b[0]) + abs(b[1])), key=lambda b: macro(yy, lp + [0, *b]))

# C's stability check: tune on 4/5 of the OOF rows, score on the other 1/5
gains = []
for a, b in StratifiedKFold(5, shuffle=True, random_state=0).split(logp, y):
    gains.append(macro(y[b], logp[b] + [0, *tune(logp[a], y[a], 0.1)]) - macro(y[b], logp[b]))
OFFSET = np.array([0, *tune(logp, y, 0.05)]) if np.mean(gains) > 0 else np.zeros(3)

pred = (logt + OFFSET).argmax(1)
sub = pd.DataFrame({"id": test.id, "label": np.array(LABELS)[pred]})
assert len(sub) == 6000 and sub.id.is_unique and set(sub.id) == set(test.id) and sub.label.isin(LABELS).all()
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"blend {({n: float(w) for n, w in zip(names, W)})} | OOF macro-F1 raw {macro(y, logp):.4f} -> tuned {macro(y, logp + OFFSET):.4f} | "
      f"offsets JB {OFFSET[1]:+.2f} PI {OFFSET[2]:+.2f} (held-out gain {np.mean(gains):+.4f}) | "
      f"test share {(np.bincount(pred, minlength=3) / len(pred)).round(3).tolist()}")